# 3. Plotting, analysis and STAC interoperability

The client ships four plots, all drawn against a country basemap:
`plot_coverage()`, `plot_timeline()`, `plot_providers()` and
`Scene.plot_footprint()`. Each returns a matplotlib `Axes` and accepts `ax=`, so
they compose into your own figures.

The backend is also a real STAC catalog, so standard tooling reads it with or
without this client.

```bash
pip install 'open-sar-triad[notebooks]'
```

In [ ]:
# Point at the hosted API, or a local build for testing.
#   BASE = 'http://localhost:8000/api/v1'   # after: python3 -m http.server 8000
BASE = None

from opensartriad import Catalog

cat = Catalog(BASE) if BASE else Catalog()
cat

## Into pandas

`to_dataframe()` gives the index-level fields, which is enough for most analysis
and does not trigger the lazy per-provider fetch.

In [ ]:
df = cat.all().to_dataframe()
print(df.shape)
df.head()

In [ ]:
df.groupby('provider').size().sort_values(ascending=False)

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

cat.all().plot_providers()
plt.show()

## Plotting

The client ships plots for the usual questions, so none of this is hand-rolled.
They need matplotlib: `pip install 'open-sar-triad[plot]'`.

Every plot returns a matplotlib `Axes` and accepts `ax=`, so they compose into
your own figures.

In [ ]:
scenes = cat.all()
scenes.plot_timeline(freq='month')
plt.show()

## Sensor modes by provider

In [ ]:
pivot = df.groupby(['provider', 'mode']).size().unstack(fill_value=0)
pivot

## Where the scenes are

`plot_coverage()` draws footprints on a country basemap, coloured by provider.

The basemap is the same world-atlas geometry the web map uses, decoded here rather
than pulling in cartopy or contextily. It is cached after the first call, and if
every mirror is blocked the scenes still plot, just without the outlines.

In [ ]:
scenes.plot_coverage()
plt.show()

### Zooming in

Pass a `bbox` to focus on an area, and `footprints=True` for true acquisition
polygons instead of bounding boxes (this fetches the full records, so it is slower).

In [ ]:
europe = cat.search(bbox=(-10, 35, 30, 60))
print(europe)
europe.plot_coverage(bbox=(-10, 35, 30, 60), footprints=True)
plt.show()

### Composing your own figure

Because each plot returns an `Axes` and accepts `ax=`, they drop into subplots.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(13, 9))
scenes.plot_coverage(ax=axes[0])
scenes.plot_timeline(ax=axes[1], freq='year')
plt.tight_layout()
plt.show()

### A single scene

In [ ]:
s = cat.search(providers='umbra', limit=1)[0]
s.plot_footprint(pad=12)
plt.show()

## Which products are available

How many scenes can satisfy each family. This is the honest answer to
'can I get complex data here?'

In [ ]:
from opensartriad import FAMILIES

rows = []
for fam in FAMILIES:
    hits = cat.search(family=fam)
    by_p = {}
    for s in hits:
        by_p[s.provider] = by_p.get(s.provider, 0) + 1
    rows.append({'family': fam, 'total': len(hits), **by_p})

import pandas as pd
pd.DataFrame(rows).set_index('family').fillna(0).astype(int)

## STAC interoperability

Search results convert straight to a STAC ItemCollection.

In [ ]:
sel = cat.search(bbox=(5.9, 47.2, 10.5, 55.1), start='2025-01-01', limit=5)
ic = sel.to_stac()

print(ic['type'], '| stac_version', ic['stac_version'], '|', len(ic['features']), 'items')
item = ic['features'][0]
print('id      :', item['id'])
print('assets  :', list(item['assets'])[:6])
print('bbox    :', item['bbox'])

### Using pystac directly

No client needed. Any STAC tool can read the catalog by URL.

```python
import pystac

root = pystac.Catalog.from_file(
    'https://www.pmuguda.com/open-sar-triad/api/v1/catalog.json'
)
for child in root.get_children():
    print(child.id, '-', child.title)
```

## Licence

Scene metadata is CC-BY 4.0. Credit the originating provider in published work.

In [ ]:
print(cat.license()['attribution'])